# Build your Own

```mermaid
classDiagram
    class Module {
        +forward(x)
        +backward(grad)
        +parameters()
        +train()
        +eval()
    }
```

In [2]:
import math
import random

class Module:
    def __init__(self):
        self.training = True

    def forward(self, x):
        raise NotImplementedError

    def backward(self, grad):
        raise NotImplementedError

    def train(self):
        self.training = True

    def eval(self):
        self.training = False

        

```mermaid
classDiagram
    class Linear {
        -weights
        -biases
        +forward(x)
        +backward(grad)
    }
```

In [3]:
class Linear(Module):
    def __init__(self, in_feats, out_feats):
        super().__init__()

        std = math.sqrt(2.0 / in_feats)

        self.weights = [
            [random.gauss(0, std) for _ in range(in_feats)]
            for _ in range(out_feats)
        ]
        self.biases = [0.0] * out_feats
        self.weight_grads = [
            [0.0] * in_feats
            for _ in range(out_feats)
        ]
        self.bias_grads = [0.0] * out_feats
        self.in_feats = in_feats
        self.out_feats = out_feats
        self.input = None

    def forward(self, x):
        self.input = x
        output = []
        for i in range(self.out_feats):
            val = self.biases[i]
            val += sum(self.weights[i][j] * x[j] for j in range(self.in_feats))
            output.append(val)
        return output

    def backward(self, grad):
        input_grad = [0.0] * self.in_feats
        for i in range(self.out_feats):
            self.bias_grads[i] += grad[i]
            for j in range(self.in_feats):
                self.weight_grads[i][j] += grad[i] * self.input[j]
                input_grad[j] += grad[i] * self.weights[i][j]
        return input_grad

    def parameters(self):
        params = []
        for i in range(self.out_feats):
            for j in range(self.in_feats):
                params.append((self.weights, i, j, self.weight_grads))
            params.append((self.biases, i, None, self.bias_grads))
        
        return params


In [4]:
class ReLU(Module):
    def __init__(self):
        super().__init__()

    def forward(self, x):
        self.mask = [1.0 if v > 0 else 0.0 for v in x]
        return [max(0, i) for i in x]
        
    def backward(self, grad):
        return [g * m for g, m in zip(grad, self.mask)]


In [5]:
class Sigmoid(Module):
    def __init__(self):
        super().__init__()
        self.output = None

    def forward(self, x):
        self.output = [
            1.0 / (1.0 + math.exp(-max(-500, min(500, v)))) for v in x
        ]
        return self.output

    def backward(self, grad):
        return [g * o * (1 - o) for g, o in zip(grad, self.output)]

In [6]:
class Tanh(Module):
    def __init__(self):
        super().__init__()
        self.output = None

    def forward(self, x):
        self.output = [math.tanh(v) for v in x]

    def backward(self, grad):
        return [g * (1 - o ** 2) for g, o in zip(grad, self.output)]

In [7]:
class Dropout(Module):
    def __init__(self, p=0.5):
        super().__init__()
        self.p = p
        self.mask = None

    def forward(self, x):
        if not self.training:
            return x
        self.mask = [0.0 if random.random() < self.p else 1.0 / (1 - self.p) for _ in x]
        return [v * m for v, m in zip(x, self.mask)]

    def backward(self, grad):
        if self.mask is None:
            return grad
        return [g * m for g, m in zip(grad, self.mask)]

In [8]:
class BatchNorm(Module):
    def __init__(self, size, momentum=0.1, eps=1e-5):
        super().__init__()
        self.size = size
        self.gamma = [1.0] * size
        self.beta = [0.0] * size
        self.gamma_grads = [0.0] * size
        self.beta_grads = [0.0] * size
        self.running_mean = [0.0] * size
        self.running_var = [1.0] * size
        self.momentum = momentum
        self.eps = eps
        self.x_norm = None
        self.std_inv = None
        self.batch_input = None

    def forward_batch(self, batch):
        batch_size = len(batch)
        output_batch = []

        if self.training:
            mean = [0.0] * self.size
            for sample in batch:
                for j in range(self.size):
                    mean[j] += sample[j]
            mean = [m / batch_size for m in mean]

            var = [0.0] * self.size
            for sample in batch:
                for j in range(self.size):
                    var[j] += (sample[j] - mean[j]) ** 2
            var = [v / batch_size for v in var]

            self.std_inv = [1.0 / math.sqrt(v + self.eps) for v in var]

            self.x_norm = []
            self.batch_input = batch

            for sample in batch:
                normed = [
                    (sample[j] - mean[j]) * self.std_inv[j]
                    for j in range(self.size)
                ]
                self.x_norm.append(normed)

                output = [
                    self.gamma[j] * normed[j] + self.beta[j]
                    for j in range(self.size)
                ]
                output_batch.append(output)

            for j in range(self.size):
                self.running_mean[j] = (1 - self.momentum) * self.running_mean[j] + self.momentum * mean[j]
                self.running_var[j] = (1 - self.momentum) * self.running_var[j] + self.momentum * var[j]

        else:
            std_inv = [1.0 / math.sqrt(v + self.eps) for v in self.running_var]
            for sample in batch:
                normed = [(sample[j] - self.running_mean[j]) * std_inv[j] for j in range(self.size)]
                output = [self.gamma[j] * normed[j] + self.beta[j] for j in range(self.size)]
                output_batch.append(output)

        return output_batch

    def forward(self, x):
        if self.training:
            for j in range(self.size):
                self.running_mean[j] = (1 - self.momentum) * self.running_mean[j] + self.momentum * x[j]

            self.std_inv = [1.0 / math.sqrt(v + self.eps) for v in self.running_var]
            self.x_norm = [(x[j] - self.running_mean[j]) * self.std_inv[j] for j in range(self.size)]
            return [self.gamma[j] * self.x_norm[j] + self.beta[j] for j in range(self.size)]
        else:
            std_inv = [1.0 / math.sqrt(v + self.eps) for v in self.running_var]
            normed = [(x[j] - self.running_mean[j]) * std_inv[j] for j in range(self.size)]
            return [self.gamma[j] * normed[j] + self.beta[j] for j in range(self.size)]

    def backward(self, grad):
        if self.x_norm is None:
            return grad
        
        x_norm = self.x_norm if not isinstance(self.x_norm[0], list) else self.x_norm[0]
        for j in range(self.size):
            self.gamma_grads[j] += x_norm[j] * grad[j]
            self.beta_grads[j] += grad[j]

        return [grad[j] * self.gamma[j] * self.std_inv[j] for j in range(self.size)]

    def parameters(self):
        params = []
        for j in range(self.size):
            params.append((self.gamma, j, None, self.gamma_grads))
            params.append((self.beta, j, None, self.beta_grads))

        return params

In [10]:
class Sequential(Module):
    def __init__(self, *modules):
        super().__init__()
        self.modules = list(modules)

    def forward(self, x):
        for m in self.modules:
            x = m.forward(x)
        return x

    def backward(self, grad):
        for m in reversed(self.modules):
            grad = m.backward(grad)
        return grad

    def parameters(self):
        params = []
        for m in self.modules:
            params.extend(m.parameters())
        return params
        
    def train(self):
        self.training = True
        for m in self.modules:
            m.train()

    def eval(self):
        self.training = False
        for m in self.modules:
            m.eval()

    def count_parameters(self):
        return len(self.parameters())


In [11]:
class MSELoss:
    def __call__(self, predicted, target):
        return sum((p - t) ** 2 for p, t in zip(predicted, target)) / len(predicted)

    def backward(self, predicted, target):
        return [2 * (p - t) / len(predicted) for p, t in zip(predicted, target)]
    

In [12]:
class BCELoss:
    def __call__(self, predicted, target):
        return sum(-(t * math.log(p) + (1 - t) * math.log(1 - p)) for p, t in zip(self.__fix_predict(predicted), target)) / len(predicted)

    def backward(self, predicted, target):
        return [((p - t) / (p * (1 - p))) for p, t in zip(self.__fix_predict(predicted), target)]

    def __fix_predict(self, p):
        eps = 1e-7
        return [max(eps, min(1 - eps, p)) for p in p]


In [13]:
class SGD:
    def __init__(self, params, lr=0.01):
        self.params = params
        self.lr = lr

    def step(self):
        for container, i, j, grad_container in self.params:
            if j is not None:
                container[i][j] -= self.lr * grad_container[i][j]
            else:
                container[i] -= self.lr * grad_container[i]

    def zero_grad(self):
        for _, i, j, grad_container in self.params:
            if j is not None:
                grad_container[i][j] = 0.0
            else:
                grad_container[i] = 0.0


In [ ]:
class Adam:
    def __init__(self, params, lr=0.001, beta1=0.9, beta2=0.999, eps=1e-8):
        self.params = params
        self.lr = lr
        self.beta1= beta1
        self.beta2 = beta2
        self.eps = eps
        self.t = 0
        self.m = [0.0] * len(params)
        self.v = [0.0] * len(params)

    def step(self):
        self.t += 1
        for idx, (container, i, j, grad_container) in enumerate(self.params):
            if j is not None:
                g = grad_container[i][j]
            else:
                g = grad_container[i]
            
            self.m[idx] = self.beta1 * self.m[idx] + (1 - self.beta1) * g
            self.v[idx] = self.beta2 * self.v[idx] + (1 - self.beta2) * g ** 2

            m_hat = self.m[idx] / (1 - self.beta1 ** self.t)
            v_hat = self.v[idx] / (1 - self.beta2 ** self.t)

            if j is not None:
                container[i][j] -= self.lr * m_hat / (math.sqrt(v_hat) + self.eps)
            else:
                container[i] -= self.lr * m_hat / (math.sqrt(v_hat) + self.eps)

    def zero_grad(self):
        for _, i, j, grad_container in self.params:
            if j is not None:
                grad_container[i][j] = 0.0
            else:
                grad_container[i] = 0.0
                